# **Preliminary Data Analysis**
*Eli Torrey, Ken Hoang, Alex Martinez*

This code block is just the initial data cleaning

In [7]:
# @title
import pandas as pd
import numpy as np
import sklearn as sk
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
import statsmodels.formula.api as smf

## ***Data cleaning***


speed_dating_df = pd.read_csv('csv_result-speeddating.csv')
speed_dating_df_copy = speed_dating_df.copy()
# Got the column names of those that had the bracket values
d_columns= speed_dating_df.loc[:, speed_dating_df.columns.str.startswith('d_')].iloc[:, 1:].columns
speed_dating_df = speed_dating_df.drop(columns=d_columns)
# Columns of partner preferences --> dropping them because we will just like from the perspective of one student
pref_o_columns = speed_dating_df.loc[:, speed_dating_df.columns.str.startswith('pref_o')].columns
# Drop these columns
speed_dating_df = speed_dating_df.drop(columns=pref_o_columns)
speed_dating_df = speed_dating_df.drop(columns=['decision_o', 'expected_happy_with_sd_people', 'age'])
# Drop columns about hobby we will just use the interest correlations columns
hobby_columns = speed_dating_df.iloc[:, 36:52].columns
speed_dating_df = speed_dating_df.drop(columns=hobby_columns)
# Drop columns about partners evalution --> again we will only look from one perspective
partner_eval_columns = speed_dating_df.loc[:, speed_dating_df.columns.str.endswith('_o')].iloc[:, 2:]
speed_dating_df = speed_dating_df.drop(columns=partner_eval_columns)
# Drop columns about thoughts of self, not necessary
speed_dating_df = speed_dating_df.drop(columns=['id', 'wave', 'attractive','sincere', 'intelligence', 'funny', 'ambition', 'met', 'match', 'expected_num_interested_in_me','expected_num_matches', 'guess_prob_liked', 'sports', 'has_null'])
# Drop all rows in demographics category that have null values
null_race_rows = speed_dating_df[speed_dating_df['race'] == '?'].index
speed_dating_df = speed_dating_df.drop(null_race_rows)
null_age_o_rows = speed_dating_df[speed_dating_df['age_o'] == '?'].index
speed_dating_df = speed_dating_df.drop(null_age_o_rows)

speed_dating_df['age_o'] = speed_dating_df['age_o'].astype(int)

# Changing all preference values to floats while dropping those with null
preference_cols = speed_dating_df.loc[:, speed_dating_df.columns.str.contains('important', case=False)].columns
for preference in preference_cols:
    null_index = speed_dating_df[speed_dating_df[preference] == '?'].index
    speed_dating_df = speed_dating_df.drop(null_index)

for preference in preference_cols:
    speed_dating_df[preference] = speed_dating_df[preference].astype(float)

# Changing all evaluation values to floats while dropping those with null
eval_of_partner_cols = speed_dating_df.loc[:, speed_dating_df.columns.str.contains('partner', case=False)].columns
for rating in eval_of_partner_cols:
    null_index = speed_dating_df[speed_dating_df[rating] == '?'].index
    speed_dating_df = speed_dating_df.drop(null_index)

for rating in eval_of_partner_cols:
    speed_dating_df[rating] = speed_dating_df[rating].astype(float)

# Changing all miscellaneous values to floats while dropping those with null

misc_cols = ['importance_same_race', 'importance_same_religion', 'interests_correlate', 'like']
for misc in misc_cols:
    null_index = speed_dating_df[speed_dating_df[misc] == '?'].index
    speed_dating_df = speed_dating_df.drop(null_index)

for misc in misc_cols:
    speed_dating_df[misc] = speed_dating_df[misc].astype(float)

na_index = speed_dating_df[speed_dating_df['decision'].isna()].index
speed_dating_df = speed_dating_df.drop(na_index)

/tmp/ipykernel_486/1255925708.py:12: DtypeWarning: Columns (4,11,12,16,17,18,19,20,40,41,42,43,44,45,52,53,54,55,56,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,108,110) have mixed types. Specify dtype option on import or set low_memory=False.
  speed_dating_df = pd.read_csv('csv_result-speeddating.csv')


# Research Question 1: How do the students' preferences correlate/compare to their decisions?

One of the most valuable aspects of this dataset is the inclusion of the level of preference (a rating 1-10) participants have towards certain traits. We wanted to measure how these preference do/don't align with the decisions ("yes"/"no" to a second date) participants ultimately made.

## → Logistic regression models

To evaluate the correlation between the traits that participants stated as preferential and their decisions, we trained two logistic regression models. Both models use the trait preference data, but they differ slightly in how that data is fed to the models and applied.  



### Model 1

This model is trained on the raw partner ratings: how participants rated their date on each trait (e.g., attractiveness, sincerity).

The goal is to establish a baseline for how well these ratings predict whether a participant says yes to a second date, and to see which traits are most strongly associated with that decision in practice.

 ### Model 2

This model applies a **weight**- to each trait’s partner rating, based on how important that trait was said to be. The goal is to measure how much a trait *actually* matters to a person relative to the other traits.

For each participant, a trait’s weight is its stated importance rating (1-10) divided by the sum of that participant’s importance ratings across all traits, so a participant’s weights sum to 1.

Each partner rating for a trait is then multiplied by that trait’s weight, giving the weighted score. These weighted scores are the features used to train the weighted logistic regression, which is compared against Model 1 (trained on the raw partner ratings) over 100 random train/test splits to see whether stated preferences improve prediction of the decision.


In [11]:
speed_dating_df['decision'].value_counts(normalize=True)

,proportion
decision,
0,0.567718
1,0.432282


In [18]:
# Separate data by preference and evaluation columns
pref_cols = speed_dating_df.columns[speed_dating_df.columns.str.endswith('_important')]
eval_cols = speed_dating_df.columns[speed_dating_df.columns.str.endswith('_partner')]

# Weight the evaluation columns based on how much the feature was stated to be important
weights = speed_dating_df[pref_cols].div(speed_dating_df[pref_cols].sum(axis=1), axis=0)
weighted_scores = weights.values * speed_dating_df[eval_cols]

# accuracies based on unweighted (n) or weighted (w) features - this will be
# important for comparing weighted and unweighted models (as a method of
# verifying the impact of stated preferences)
average_accuracy_n = 0
average_accuracy_w = 0

total_iterations = 100

for i in range(total_iterations):
    # Split the data by index so that it remains consistent among both models
    train_idx, test_idx = train_test_split(speed_dating_df.index, test_size=0.2)

    # Features for each model
    X_w = weighted_scores
    X_n = speed_dating_df[eval_cols]
    y = speed_dating_df['decision']

    # Train the first model, based on raw ratings
    nonweighted_model = sk.linear_model.LogisticRegression().fit(X_n.loc[train_idx], y.loc[train_idx])
    nonweighted_accuracy = nonweighted_model.score(X_n.loc[test_idx], y[test_idx])

    # Train the second model, based on the weighted scores
    weighted_model = sk.linear_model.LogisticRegression().fit(X_w.loc[train_idx], y.loc[train_idx])
    weighted_accuracy = weighted_model.score(X_w.loc[test_idx], y[test_idx])


    average_accuracy_w += weighted_accuracy
    average_accuracy_n += nonweighted_accuracy

average_accuracy_w /= total_iterations
average_accuracy_n /= total_iterations

print(f'Average Weighted Accuracy: {average_accuracy_w:.1%}')
print(f'Average Nonweighted Accuracy: {average_accuracy_n:.1%}')

Average Weighted Accuracy: 71.3%
Average Nonweighted Accuracy: 74.8%


**Takeaways:** The first model (raw partner ratings; unweighted) predicts decisions with an average accuracy of 74.8%, compared with 71.4% for the weighted model, across 100 random train/test splits. Weighting each rating by its stated importance didn't improve prediction and in fact made it worse by a little more than 3%. This suggests that stated preferences add no predictive information beyond what the ratings already capture, and that what people say they value isn't a better guide to their decisions than how they actually rated their date.

# Research Question 2: How are different factors correlated with match success, and which factor holds the most weight?

Our strategy to answer this question consists of just one model this time.

The model is trained using the partner ratings to predict whether a participant says yes to a second date.

Each rating is first standardized to put them on the same scale. This way we can compare their coefficients fairly.

The trait coefficients show how strongly each trait is associated with a “yes” decision when the other traits are held constant. The sign of the coeffeicient represents the direction in which the factor relates to match success (ie. positively or negatively), and magnitudes tells us which factors matter more or less than others.

The coefficients only represent association (not causation).

In [19]:
from sklearn.preprocessing import StandardScaler

# Train model on whole dataset
features = speed_dating_df[eval_cols]
outcome = speed_dating_df['decision']

# Standardizing the scale for each feature because the standard deviation between each feature varied widely
# Meaning that a one point jump in different features are not all equal
X, y = StandardScaler().fit_transform(features), outcome
rating_model = sk.linear_model.LogisticRegression().fit(X, y)
coefficients = rating_model.coef_[0]

for i in range(len(eval_cols)):
    print(f'Weight of {eval_cols[i]}: {coefficients[i]:.3f}')


Weight of attractive_partner: 1.064
Weight of sincere_partner: -0.187
Weight of intelligence_partner: 0.032
Weight of funny_partner: 0.520
Weight of ambition_partner: -0.294
Weight of shared_interests_partner: 0.560


**Takeaways:** Attractiveness is the trait most strongly associated with a yes decision: a one-standard-deviation increase in a partner’s attractiveness rating roughly triples the odds of a yes, holding the other traits constant. Shared interests and funniness follow, each increasing the odds by about 70%. Intelligence shows essentially no association once the other traits are accounted for. Sincerity and ambition have small negative coefficients, meaning that among partners rated similarly on the other traits, those rated as more sincere or ambitious were slightly less likely to receive a yes.

# Research Question 3: Are there certain demographics (race, gender, age)that are disproportionately rated lower?

This analysis compares how partners are rated across demographics, in this case specifically racial groups. It groups the data by race_o (which is the race of the person being rated, meaning the person opposite the participant giving the ratings) and averages each trait rating, the overall like rating, and the decision column. Since decision is represented by a 0 ("no") or a 1 ("yes"), its average is the share of “yes” decisions each group received.

For each rating, we display the groups with the highest and lowest averages, as well as the full table in order to show the size of the gaps. Groups who  demonstrate low ratings across a variety of traits and final decisions would seem to be rated lower systematically.

In [20]:
# **Focus on Research Question 3**
rate_cols = list(eval_cols) + ['like', 'decision']
rate_cols

average_rate_by_race = speed_dating_df.groupby('race_o').mean(numeric_only=True).loc[:, rate_cols]

for element in rate_cols:
    p = average_rate_by_race[element].agg(['idxmax', 'idxmin'])
    print(f'Max {element}: {p['idxmax']}')
    print(f'Min {element}: {p['idxmin']} \n')

average_rate_by_race

Max attractive_partner: Latino/Hispanic American
Min attractive_partner: Asian/Pacific Islander/Asian-American 

Max sincere_partner: Latino/Hispanic American
Min sincere_partner: European/Caucasian-American 

Max intelligence_partner: Other
Min intelligence_partner: Black/African American 

Max funny_partner: Latino/Hispanic American
Min funny_partner: Asian/Pacific Islander/Asian-American 

Max ambition_partner: Black/African American
Min ambition_partner: Asian/Pacific Islander/Asian-American 

Max shared_interests_partner: Latino/Hispanic American
Min shared_interests_partner: Asian/Pacific Islander/Asian-American 

Max like: Latino/Hispanic American
Min like: Asian/Pacific Islander/Asian-American 

Max decision: European/Caucasian-American
Min decision: Asian/Pacific Islander/Asian-American 



,attractive_partner,sincere_partner,intelligence_partner,funny_partner,ambition_partner,shared_interests_partner,like,decision
race_o,,,,,,,,
Asian/Pacific Islander/Asian-American,5.715543,7.204614,7.412568,6.014876,6.758349,5.115361,5.864602,0.355798
Black/African American,6.140845,7.222535,7.304225,6.712676,6.836620,5.467606,6.133803,0.428169
European/Caucasian-American,6.398899,7.130047,7.365889,6.468799,6.773597,5.586786,6.220687,0.464604
Latino/Hispanic American,6.453971,7.334838,7.354693,6.855596,6.786101,5.716606,6.448556,0.462094
Other,6.020225,7.298876,7.420225,6.539326,6.811236,5.522472,6.129213,0.404494


**Takeaways:** Asian/Pacific Islander/Asian-American partners have the lowest average rating on four of the six traits (attractiveness, fun, ambition, shared interests), as well as on like and decision. Their “yes” rate is around 36%, whereas European/Caucasian-American and Latino/Hispanic American partners had a "yes" rate of around 46%. Differences on sincerity and intelligence are small across all groups. The largest gaps appear in attractiveness, fun, shared interests, and decision. These gaps suggest the lower ratings for Asian/Pacific Islanders show a general rating bias rather than being single trait-specific.

## Research Question 4: Are there demographics that perform well in the ratings but seem to earn fewer matches?

Here we test whether a partner’s race affects participant decisions, over and above how they were rated. It trains a logistic regression on the full dataset, using the standardized trait ratings plus dummy variables for the partner’s race (race_o), with Asian/Pacific Islander as the reference category.

Because the trait ratings are included in the model, each race coefficient shows how much more or less likely a “yes” is for that group compared to the reference group, among partners who received the same ratings. A group with a negative coefficient earns fewer matches than its ratings would predict, which is the pattern the research question asks about. A coefficient near zero means decisions track ratings equally across groups.

These coefficients show association, not causation. Also, the estimates produced from group fewer participants could be viewed as unstable because of the smaller sample size.

In [21]:
evals_std = pd.DataFrame(StandardScaler().fit_transform(speed_dating_df[eval_cols]), columns=eval_cols, index=speed_dating_df.index)
race_dummies = pd.get_dummies(speed_dating_df['race_o'], prefix='race', drop_first=True, dtype=int)

X = pd.concat([evals_std, race_dummies], axis=1)
y = speed_dating_df['decision']

model_demographic = sk.linear_model.LogisticRegression().fit(X, y)
coef_dict = {name: float(coef) for name, coef in zip(X.columns, model_demographic.coef_[0])}
coef_dict['Asian/Pacific Islander'] = 0.0

coef_dict

{'attractive_partner': 1.05827923237306,
 'sincere_partner': -0.18321997315239244,
 'intelligence_partner': 0.028075382025733016,
 'funny_partner': 0.5246186225473327,
 'ambition_partner': -0.2923729611503241,
 'shared_interests_partner': 0.5582120301035225,
 'race_Black/African American': -0.04138545663457602,
 'race_European/Caucasian-American': 0.03432430686865525,
 'race_Latino/Hispanic American': -0.09326601169359462,
 'race_Other': -0.048191533060156756,
 'Asian/Pacific Islander': 0.0}

**Takeaways:** Because Asian/Pacific Islander participants had the lowest ratings on most traits, we used them as the baseline group in a model that predicts decisions from the trait ratings plus race. The race coefficients were small (between about -0.09 and +0.03), especially compared with the trait coefficients such as attractiveness (1.06).

This suggests that among partners who received similar ratings, decision rates stayed similar across races. In other words, race adds little predictive information once the ratings are known.

This does not rule out bias in the ratings themselves, however. We didn't test whether the small differences are statistically significant.